# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

### The Target: April `missed_clicks` (Regression)
We are predicting April `missed_clicks`. This gives us a continuous output that inherently serves as the ranking score for our final review queue. To compute this, we use **April's own tier average CTR**, not March's averages. Judging a page against its own month's peers protects the target from platform-wide CTR shifts (like seasonality or algorithm updates) that would unfairly penalize everyone if judged against stale baselines.

### Model Sequence
We will build a simple, readable **Linear Regression** model first. This lets us read the directional weights to verify they make real-world sense before jumping to a complex "black box" like a Random Forest.

### Features and Leakage
We are using 5 March-only signals. These are 100% safe from target leakage. The rule isn't just about "different months"; it's about being **available at the decision moment**. Since an analyst on May 1st has full access to finalized March data, these features are settled facts and do not sneak in any future information about the April outcome.

### Eligible Population
We filter for **>= 500 impressions in both March and April**. In W03/W04 we proved that CTR on < 500 impressions is statistically noisy. Since our April target relies on an April CTR, allowing pages with low April volume would mean evaluating the model against a garbage "answer key."

In [2]:
import duckdb
from dotenv import load_dotenv
import os

# 1. Connect to DuckDB and load the Hugging Face token from your .env file
con = duckdb.connect()
load_dotenv()
HF_TOKEN = os.getenv('HF_TOKEN')
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

# The path to the daily performance table in the warehouse
FACT_DAILY = "read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet')"

# 2. Extract April's data with strict date boundaries and client connectivity filters
april_query = f"""
    SELECT
        content_hash_id,
        client_hash_id,
        SUM(gsc_impressions)        AS total_impressions,
        SUM(gsc_clicks)             AS total_clicks,
        AVG(gsc_avg_position)       AS average_position,
        SUM(ga4_sessions)           AS total_sessions,
        SUM(ga4_engaged_sessions)   AS total_engaged_sessions,
        SUM(ga4_pageviews)          AS total_pageviews
    FROM {FACT_DAILY}
    WHERE report_date >= '2026-04-01' AND report_date < '2026-05-01'
      AND client_has_ga4 IS TRUE
      AND client_has_gsc IS TRUE
    GROUP BY content_hash_id, client_hash_id
"""

# 3. Execute the query and load it into a pandas DataFrame
print("Querying April data from Hugging Face... (this may take a moment)")
april_df = con.sql(april_query).df()

# Verify what we pulled
print(f"April Data Shape: {april_df.shape[0]:,} rows, {april_df.shape[1]} columns")
print(april_df.head())


Querying April data from Hugging Face... (this may take a moment)
April Data Shape: 285,820 rows, 8 columns
            content_hash_id           client_hash_id  total_impressions  \
0  content_b339cfee19d127a0  client_3ffa76342f366962                0.0   
1  content_6e17aca1f2bc44c6  client_3ffa76342f366962                0.0   
2  content_c3e88fb024417c66  client_3ffa76342f366962                5.0   
3  content_2bc968c4d9133dc0  client_3ffa76342f366962                0.0   
4  content_5856d3677af99cc5  client_3ffa76342f366962                0.0   

   total_clicks  average_position  total_sessions  total_engaged_sessions  \
0           0.0               NaN             0.0                     0.0   
1           0.0               NaN             0.0                     0.0   
2           0.0         26.111111             0.0                     0.0   
3           0.0               NaN             0.0                     0.0   
4           0.0               NaN             0.0       

In [3]:
# 1. Count unique clients in our April dataframe
april_client_count = april_df['client_hash_id'].nunique()

# 2. Query DuckDB to get the unique client count for March for comparison
march_client_query = f"""
    SELECT COUNT(DISTINCT client_hash_id)
    FROM {FACT_DAILY}
    WHERE report_date >= '2026-03-01' AND report_date < '2026-04-01'
      AND client_has_ga4 IS TRUE
      AND client_has_gsc IS TRUE
"""
march_client_count = con.sql(march_client_query).fetchone()[0]

print(f"Unique Clients in April: {april_client_count}")
print(f"Unique Clients in March: {march_client_count}")

# 3. Let's go one step further and verify if they are the EXACT same clients
march_clients_query = f"""
    SELECT DISTINCT client_hash_id
    FROM {FACT_DAILY}
    WHERE report_date >= '2026-03-01' AND report_date < '2026-04-01'
      AND client_has_ga4 IS TRUE
      AND client_has_gsc IS TRUE
"""
march_clients = set([row[0] for row in con.sql(march_clients_query).fetchall()])
april_clients = set(april_df['client_hash_id'].unique())

print(f"Clients dropped (in March but NOT in April): {len(march_clients - april_clients)}")
print(f"New clients (in April but NOT in March): {len(april_clients - march_clients)}")


Unique Clients in April: 48
Unique Clients in March: 43
Clients dropped (in March but NOT in April): 0
New clients (in April but NOT in March): 5


An inner join is used because the model requires March features to make a prediction; the 5 clients that came online in April (with no March history) are correctly excluded, and no existing client's pages were lost due to connectivity changes between the two months


In [4]:
import pandas as pd

# 1. First, extract the March features from the warehouse
march_query = f"""
    SELECT
        content_hash_id,
        client_hash_id,
        SUM(gsc_impressions)        AS total_impressions,
        SUM(gsc_clicks)             AS total_clicks,
        AVG(gsc_avg_position)       AS average_position,
        SUM(ga4_sessions)           AS total_sessions,
        SUM(ga4_engaged_sessions)   AS total_engaged_sessions,
        SUM(ga4_pageviews)          AS total_pageviews
    FROM {FACT_DAILY}
    WHERE report_date >= '2026-03-01' AND report_date < '2026-04-01'
      AND client_has_ga4 IS TRUE
      AND client_has_gsc IS TRUE
    GROUP BY content_hash_id, client_hash_id
"""
print("Querying March data...")
monthly_features = con.sql(march_query).df()

# 2. Now perform the inner join to keep only pages that existed in BOTH months
merged_df = pd.merge(
    monthly_features, 
    april_df, 
    how="inner", 
    on=["client_hash_id", "content_hash_id"], 
    suffixes=('_march', '_april')
)

# Verify the merge
print(f"March rows: {monthly_features.shape[0]:,}")
print(f"April rows: {april_df.shape[0]:,}")
print(f"Merged (Intersection) rows: {merged_df.shape[0]:,}")
print("\nNew columns:")
print(merged_df.columns.tolist())
march_ids = set(monthly_features["content_hash_id"])
merged_ids = set(merged_df["content_hash_id"])
missing = march_ids - merged_ids
print(missing)

Querying March data...
March rows: 260,737
April rows: 285,820
Merged (Intersection) rows: 260,736

New columns:
['content_hash_id', 'client_hash_id', 'total_impressions_march', 'total_clicks_march', 'average_position_march', 'total_sessions_march', 'total_engaged_sessions_march', 'total_pageviews_march', 'total_impressions_april', 'total_clicks_april', 'average_position_april', 'total_sessions_april', 'total_engaged_sessions_april', 'total_pageviews_april']
{'content_0be895ebbcaab26d'}


In [5]:
april_df[april_df['content_hash_id']=='content_0be895ebbcaab26d']

,content_hash_id,client_hash_id,total_impressions,total_clicks,average_position,total_sessions,total_engaged_sessions,total_pageviews


In [6]:
investigate_query = f"""
    SELECT 
        report_date, client_has_ga4, client_has_gsc, gsc_impressions 
    FROM {FACT_DAILY}
    WHERE content_hash_id = 'content_0be895ebbcaab26d' 
      AND report_date >= '2026-04-01' 
      AND report_date < '2026-05-01'
"""
print(con.sql(investigate_query).df())


Empty DataFrame
Columns: [report_date, client_has_ga4, client_has_gsc, gsc_impressions]
Index: []


The inner join drops exactly 1 of 260,737 March pages. Investigation confirmed this page has zero fact-table rows in April entirely (not a connectivity flag issue — the client's other pages report normally), consistent with the page being deleted or unpublished between March and April. This is expected content churn, not a data quality issue

In [7]:
import numpy as np

# 1. Apply the strict eligibility filter (>= 500 impressions in BOTH months)
eligible_df = merged_df[
    (merged_df['total_impressions_march'] >= 500) & 
    (merged_df['total_impressions_april'] >= 500)
].copy()

print(f"Eligible pages for modeling: {eligible_df.shape[0]:,}\n")

# 2. Calculate April's actual CTR (with the zero-impression NaN guard)
eligible_df['ctr_april'] = np.where(
    eligible_df['total_impressions_april'] > 0, 
    eligible_df['total_clicks_april'] / eligible_df['total_impressions_april'], 
    np.nan
)

# 3. Create April's position_tier (using the exact W04 bins/labels)
bins = [-float("inf"), 3, 10, 20, 50, float("inf")]
labels = ["top_3", "page_1", "striking", "page_3_5", "deep"]

eligible_df["position_tier_april"] = pd.cut(eligible_df["average_position_april"], bins=bins, labels=labels)
eligible_df["position_tier_april"] = eligible_df["position_tier_april"].cat.add_categories(["no_data"]).fillna("no_data")

# 4. Calculate April's tier_avg_ctr (grouping by April's own tiers)
raw_ctr_by_pos_april = eligible_df.groupby("position_tier_april", observed=False)["ctr_april"].mean()
eligible_df["tier_avg_ctr_april"] = eligible_df["position_tier_april"].map(raw_ctr_by_pos_april)

# Let's peek at the final calculated columns
print("April Math Verification:")
print(eligible_df[["content_hash_id", "total_impressions_april", "ctr_april", "position_tier_april", "tier_avg_ctr_april"]].head())


Eligible pages for modeling: 37,004

April Math Verification:
              content_hash_id  total_impressions_april  ctr_april  \
3    content_98d8996ce83fdb7d                    855.0    0.00117   
19   content_646653475082a415                    576.0    0.00000   
32   content_f3a75d8cf58dd50b                   1775.0    0.00000   
45   content_6cd0c162158858c3                   1019.0    0.00000   
123  content_4303ceb432aad51d                    949.0    0.00000   

    position_tier_april  tier_avg_ctr_april  
3              striking            0.002840  
19             striking            0.002840  
32               page_1            0.003254  
45             striking            0.002840  
123            page_3_5            0.001271  


In [8]:
eligible_df["position_tier_april"].value_counts()

position_tier_april
page_1      16639
page_3_5    10267
striking     8916
top_3         952
deep          230
no_data         0
Name: count, dtype: int64

In [9]:
# 1. Calculate the gap (How much worse did this page perform vs. its tier average?)
eligible_df['ctr_gap_april'] = eligible_df['tier_avg_ctr_april'] - eligible_df['ctr_april']

# 2. Clip at 0 (If a page BEAT its tier average, the gap is negative. We turn negative numbers into 0 so we don't accidentally calculate "negative missed clicks").
eligible_df['ctr_gap_april'] = eligible_df['ctr_gap_april'].clip(lower=0)

# 3. Multiply the gap by actual impressions to get the final raw number of missed clicks
eligible_df['missed_clicks_april'] = eligible_df['ctr_gap_april'] * eligible_df['total_impressions_april']

# Verification
print("Target created! Here are the top 5 pages by actual April missed clicks:")
print(eligible_df[['content_hash_id', 'tier_avg_ctr_april', 'ctr_april', 'ctr_gap_april', 'missed_clicks_april']].sort_values('missed_clicks_april', ascending=False).head())
eligible_df["missed_clicks_april"].describe()

Target created! Here are the top 5 pages by actual April missed clicks:
                 content_hash_id  tier_avg_ctr_april  ctr_april  \
196945  content_0e03de7680314cd5            0.005015   0.002397   
16978   content_99fc6465edb0e52c            0.002840   0.000004   
26684   content_39e19a3ec2d95f9d            0.003254   0.000011   
19784   content_62770e1299963fe4            0.003254   0.000999   
119889  content_77276ad7a26f4905            0.003254   0.001255   

        ctr_gap_april  missed_clicks_april  
196945       0.002617           799.174869  
16978        0.002837           760.126319  
26684        0.003243           606.060065  
19784        0.002255           514.370691  
119889       0.001999           454.084266  


count    37004.000000
mean         5.239647
std         16.107787
min          0.000000
25%          0.000000
50%          1.393662
75%          4.590166
max        799.174869
Name: missed_clicks_april, dtype: float64

In [10]:
# Create the ML-ready target
eligible_df['target_log_missed_clicks'] = np.log1p(eligible_df['missed_clicks_april'])

# 1. Calculate our missing contract feature: ctr_march
eligible_df['ctr_march'] = np.where(
    eligible_df['total_impressions_march'] > 0, 
    eligible_df['total_clicks_march'] / eligible_df['total_impressions_march'], 
    0.0
)

# 2. Log transform ONLY the 3 heavy-tailed volume features from our W03 contract
eligible_df['log_impressions_march'] = np.log1p(eligible_df['total_impressions_march'])
eligible_df['log_clicks_march'] = np.log1p(eligible_df['total_clicks_march'])
eligible_df['log_engaged_sessions_march'] = np.log1p(eligible_df['total_engaged_sessions_march'])

# Our final 5 ML Features (X):
# ['log_impressions_march', 'log_clicks_march', 'log_engaged_sessions_march', 'ctr_march', 'average_position_march']

print("Discipline restored. Section 1 complete.")
eligible_df['target_log_missed_clicks'].describe()

Discipline restored. Section 1 complete.


count    37004.000000
mean         1.043175
std          1.080240
min          0.000000
25%          0.000000
50%          0.872825
75%          1.721009
max          6.684830
Name: target_log_missed_clicks, dtype: float64

In [11]:
eligible_df['total_impressions_march'].min()

np.float64(500.0)

## 2. Split design

### Time-Aware Split (The Decision Point)
Our split simulates a real-world decision: using **March** data (features) to predict an **April** outcome (target). This guarantees no time-travel leakage.

### Grouped by Client
We group the train/test split by `client_hash_id`. As we discovered in W03, a single dominant client accounts for roughly 31,887 pages. A random split would bleed this client into both training and testing, allowing the model to cheat by simply memorizing that client's site structure rather than learning generalizable SEO rules.


*(Note: This is a single grouped train/test split. Results may vary depending on which clients happen to land in the test set. A full GroupKFold cross-validation would be the rigorous next step to ensure stability across clients).*

In [12]:
from sklearn.model_selection import GroupShuffleSplit

# Our locked, 5-feature contract
feature_cols = ['log_impressions_march', 'log_clicks_march', 
                'log_engaged_sessions_march', 'ctr_march', 'average_position_march']

X = eligible_df[feature_cols]
y = eligible_df['target_log_missed_clicks']
groups = eligible_df['client_hash_id']  # The boundary we refuse to split

# We use test_size=0.25, but due to the massive skew in client page counts (the "lump"), 
# this actually resulted in only 8 clients landing in the test set (5,254 pages).
# CAVEAT: 8 clients is a very small sample. A single unusual client in this test set 
# could heavily swing our final Precision@50 metric. (A full GroupKFold would fix this).
splitter = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(splitter.split(X, y, groups=groups))


X_train, X_test = X.iloc[train_idx].copy(), X.iloc[test_idx].copy()
y_train, y_test = y.iloc[train_idx].copy(), y.iloc[test_idx].copy()

# Recover the client IDs for our test set so we can prove there's no overlap
train_clients = eligible_df.iloc[train_idx]['client_hash_id']
test_clients = eligible_df.iloc[test_idx]['client_hash_id']

# THE PROOF: Assert that the intersection of training clients and testing clients is exactly zero.
assert len(set(train_clients) & set(test_clients)) == 0, "DATA LEAKAGE DETECTED: A client exists in both sets!"

print(f"Split complete. No data leakage detected.")
print(f"Training on {len(set(train_clients))} clients ({X_train.shape[0]:,} pages)")
print(f"Testing on {len(set(test_clients))} clients ({X_test.shape[0]:,} pages)")



Split complete. No data leakage detected.
Training on 22 clients (31,750 pages)
Testing on 8 clients (5,254 pages)


## 3. Train + compare vs my baseline

### The Evaluation Contract (Graded against April)
Both the baseline rule and the ML model make their predictions using ONLY March data. However, both will be evaluated against the **exact same April ground truth**. (Evaluating the March baseline against March data would be circular and trivially yield a perfect score).

We use two metrics side-by-side on the Top 50 recommended pages:
1. **Precision@50 (Binary):** What percentage of the Top 50 *actually* met the baseline rule's condition in April? (Actual April CTR < 50% of April Tier Average, with >= 500 April impressions). This reuses the exact same threshold from W04 to ensure a fair, apples-to-apples comparison.
2. **Total Captured Missed Clicks @ 50 (Business Value):** What is the sum of actual April `missed_clicks` captured by the Top 50 recommendations? This tells the business the real value of the prioritized list.


### Base Rate & Tie Policy
- **Base Rate:** Before evaluating Precision@50, we must compute the background base rate (what fraction of ALL eligible pages actually met the April opportunity condition?). This provides a baseline for random guessing, letting us know if our Precision@50 is actually impressive.
- **Tie Policy:** If the model outputs identical predicted `missed_clicks` scores for multiple pages right at the Top 50 cutoff, we will break ties by sorting alphabetically by `content_hash_id`. This prevents Pandas from making silent ranking decisions based on original row order.

In [13]:
from sklearn.linear_model import LinearRegression

# Step 1: Initialize and train the model
model = LinearRegression()
model.fit(X_train, y_train)

# Step 2: Get predictions on the test set (Remember, these are in Log Space!)
y_pred_log = model.predict(X_test)

# Step 3: The Sanity Check - Read the model's "mind"
print(f"Base Intercept: {model.intercept_:.4f}\n")
print("Feature Weights (How much 1 unit of change impacts the prediction):")
for feature, weight in zip(feature_cols, model.coef_):
    print(f"{feature}: {weight:.4f}")




Base Intercept: -4.1367

Feature Weights (How much 1 unit of change impacts the prediction):
log_impressions_march: 0.8647
log_clicks_march: -0.5632
log_engaged_sessions_march: -0.2766
ctr_march: -4.2804
average_position_march: -0.0340


average_position_march's coefficient came out slightly negative (-0.034), the opposite sign of what SEO intuition predicts (worse position should predict more missed clicks). This is likely multicollinearity — we already proved in Signal A that CTR and position are tightly correlated, so the model may be splitting position's signal awkwardly between the two overlapping features. This coefficient should not be read as 'position doesn't matter' or 'better position causes fewer missed clicks' — only that its individual weight is unreliable when ctr_march is also present

In [20]:
import numpy as np

# Step 1: Reverse the log transform so we are dealing with REAL clicks again
y_pred_real = np.expm1(y_pred_log)

# Step 2: Build our Results dataframe from the test set
results_df = eligible_df.iloc[test_idx][['content_hash_id', 'client_hash_id','missed_clicks_april', 'ctr_april', 'tier_avg_ctr_april']].copy()
results_df['predicted_missed_clicks'] = y_pred_real

# Step 3: Evaluate our two business metrics on the Top 50

# Sort descending by what the MODEL thinks is best
ranked_results = results_df.sort_values(by='predicted_missed_clicks', ascending=False)
top_50 = ranked_results.head(50).copy()

# Metric 1: Precision@50
# Create our "Ground Truth" answer key: Did it actually meet the rule in April?
top_50['is_actual_opportunity'] = top_50['ctr_april'] < (0.5 * top_50['tier_avg_ctr_april'])
precision_at_50 = top_50['is_actual_opportunity'].mean()

# Metric 2: Total Captured Missed Clicks @ 50
# Sum up the ACTUAL missed clicks (the real business value) caught in this Top 50 net
total_captured_clicks = top_50['missed_clicks_april'].sum()

print("--- ML Model Evaluation (Test Set) ---")
print(f"Precision@50: {precision_at_50 * 100:.1f}%")
print(f"Total Captured Missed Clicks @ 50: {total_captured_clicks:,.0f} clicks")


--- ML Model Evaluation (Test Set) ---
Precision@50: 96.0%
Total Captured Missed Clicks @ 50: 1,274 clicks


In [22]:
base_rate = (results_df['ctr_april'] < (0.5 * results_df['tier_avg_ctr_april'])).mean()
print(f"Base rate (all eligible test pages): {base_rate*100:.1f}%")
print(top_50['content_hash_id'])
print("\n--- Top 50 Client Breakdown ---")
print(top_50['client_hash_id'].value_counts())



Base rate (all eligible test pages): 48.6%
206238    content_66bf45eb0c5bb550
238724    content_cc9732f0da1d8d2d
165706    content_c9643f42e0fb5214
83986     content_7215022767f13b65
71944     content_335cb77b9794b587
137370    content_551a522809e0358c
49421     content_4257d1459eb1fdd1
62269     content_1bc8782404e3b132
145393    content_c1731fac6a7c5426
133294    content_8028c5009353ea97
2441      content_a85bf5efbd1f137e
145392    content_c5dc108cc7ed608e
61696     content_e7f0b8bdc0ace93a
67825     content_5cc6fa5852bf25f1
185885    content_31d2318a4c05a27b
71943     content_b333c5911a0624ff
226500    content_0bc57e8596bd0e24
246886    content_cb547c396b4bf957
115403    content_1754f29776da1940
212813    content_6d7fd655cb482ab2
204738    content_3d611a5bb49884fa
22890     content_bf5f7d6b2f69141a
51654     content_b534d074a411c288
108903    content_ec46de995069beab
66229     content_a1140fa63c13be8b
194123    content_b9e279437c729106
259129    content_fc0d3723ce5a9bba
237416    co

 The Linear Regression model successfully learned SEO signals, achieving a Precision@50 of 96% (nearly double the random guessing Base Rate of 48.6%). However, this evaluation comes with a strict caveat: 78% of the Top 50 recommendations belong to a single client.

While this is expected business behavior (the model correctly prioritized a massive client with the highest raw traffic volume and worst CTR), it means our 96% precision score is heavily weighted on the model's performance on that one specific client. A full k-fold cross-validation is required to prove the model's accuracy generalizes evenly across all clients.

In [23]:
# 1. Recreate position_tier_march
bins = [-float("inf"), 3, 10, 20, 50, float("inf")]
labels = ["top_3", "page_1", "striking", "page_3_5", "deep"]
eligible_df["position_tier_march"] = pd.cut(eligible_df["average_position_march"], bins=bins, labels=labels)
eligible_df["position_tier_march"] = eligible_df["position_tier_march"].cat.add_categories(["no_data"]).fillna("no_data")

# 2. Recreate tier_avg_ctr_march (using ONLY the clean eligible_df population)
raw_ctr_by_pos_march = eligible_df.groupby("position_tier_march", observed=False)["ctr_march"].mean()
eligible_df["tier_avg_ctr_march"] = eligible_df["position_tier_march"].map(raw_ctr_by_pos_march)

# 3. Calculate the Baseline's ranking score: missed_clicks_march
eligible_df['ctr_gap_march'] = eligible_df['tier_avg_ctr_march'] - eligible_df['ctr_march']
eligible_df['ctr_gap_march'] = eligible_df['ctr_gap_march'].clip(lower=0)
eligible_df['missed_clicks_march'] = eligible_df['ctr_gap_march'] * eligible_df['total_impressions_march']

print("Baseline logic recomputed safely on the clean population!")
print(eligible_df[['content_hash_id', 'tier_avg_ctr_march', 'ctr_march', 'missed_clicks_march']].head())


Baseline logic recomputed safely on the clean population!
              content_hash_id  tier_avg_ctr_march  ctr_march  \
3    content_98d8996ce83fdb7d            0.003676   0.000218   
19   content_646653475082a415            0.003676   0.001071   
32   content_f3a75d8cf58dd50b            0.003676   0.000495   
45   content_6cd0c162158858c3            0.003676   0.000269   
123  content_4303ceb432aad51d            0.001516   0.001041   

     missed_clicks_march  
3              15.847180  
19              7.292844  
32              6.429228  
45             12.671103  
123             0.912845  


In [24]:
# 1. Grab the exact same test set pages we used for the ML model
baseline_results = eligible_df.iloc[test_idx].copy()

# 2. Sort descending by what the BASELINE thinks is best (March's missed clicks)
ranked_baseline = baseline_results.sort_values(by='missed_clicks_march', ascending=False)
baseline_top_50 = ranked_baseline.head(50).copy()

# 3. Evaluate our two business metrics on the Baseline's Top 50

# Metric 1: Precision@50
# Did the baseline's picks actually meet the rule in April?
baseline_top_50['is_actual_opportunity'] = baseline_top_50['ctr_april'] < (0.5 * baseline_top_50['tier_avg_ctr_april'])
baseline_precision_at_50 = baseline_top_50['is_actual_opportunity'].mean()

# Metric 2: Total Captured Missed Clicks @ 50
# Sum up the ACTUAL missed clicks caught in the baseline's net
baseline_total_captured = baseline_top_50['missed_clicks_april'].sum()

print("--- BASELINE EVALUATION (Test Set) ---")
print(f"Precision@50: {baseline_precision_at_50 * 100:.1f}%")
print(f"Total Captured Missed Clicks @ 50: {baseline_total_captured:,.0f} clicks")

print("\n--- FINAL COMPARISON ---")
print(f"ML Model Captured: {total_captured_clicks:,.0f} clicks")
print(f"Baseline Captured: {baseline_total_captured:,.0f} clicks")
print(f"Winner Difference: {total_captured_clicks - baseline_total_captured:,.0f} clicks")


--- BASELINE EVALUATION (Test Set) ---
Precision@50: 94.0%
Total Captured Missed Clicks @ 50: 2,131 clicks

--- FINAL COMPARISON ---
ML Model Captured: 1,274 clicks
Baseline Captured: 2,131 clicks
Winner Difference: -856 clicks


In [26]:
from sklearn.ensemble import RandomForestRegressor
import numpy as np

# 1. Initialize and Train the more powerful model
rf_model = RandomForestRegressor(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)

# 2. Get predictions and IMMEDIATELY reverse the log transform!
rf_pred_log = rf_model.predict(X_test)
rf_pred_real = np.expm1(rf_pred_log)

# 3. Build our Results dataframe for the Random Forest
rf_results_df = eligible_df.iloc[test_idx][['content_hash_id', 'client_hash_id', 'missed_clicks_april', 'ctr_april', 'tier_avg_ctr_april']].copy()
rf_results_df['predicted_missed_clicks'] = rf_pred_real

# 4. Sort descending by what the Random Forest thinks is best
rf_ranked = rf_results_df.sort_values(by='predicted_missed_clicks', ascending=False)
rf_top_50 = rf_ranked.head(50).copy()

# 5. Evaluate our two business metrics
rf_top_50['is_actual_opportunity'] = rf_top_50['ctr_april'] < (0.5 * rf_top_50['tier_avg_ctr_april'])
rf_precision_at_50 = rf_top_50['is_actual_opportunity'].mean()
rf_total_captured = rf_top_50['missed_clicks_april'].sum()

print("--- RANDOM FOREST EVALUATION (Test Set) ---")
print(f"Precision@50: {rf_precision_at_50 * 100:.1f}%")
print(f"Total Captured Missed Clicks @ 50: {rf_total_captured:,.0f} clicks")

print("\n--- THE GRAND FINALE SCOREBOARD ---")
print(f"1. Baseline Captured:        {baseline_total_captured:,.0f} clicks")
print(f"2. Linear Reg. Captured:     {total_captured_clicks:,.0f} clicks")
print(f"3. Random Forest Captured:   {rf_total_captured:,.0f} clicks")


--- RANDOM FOREST EVALUATION (Test Set) ---
Precision@50: 86.0%
Total Captured Missed Clicks @ 50: 1,698 clicks

--- THE GRAND FINALE SCOREBOARD ---
1. Baseline Captured:        2,131 clicks
2. Linear Reg. Captured:     1,274 clicks
3. Random Forest Captured:   1,698 clicks


## 4. Errors and interpretation

### Planned Sensitivity Check: The Volume Dominance
Our target (`missed_clicks`) multiplies a CTR gap by `total_impressions`. The risk is that the model takes the lazy route: *"big pages stay big."* March `total_impressions` and `total_clicks` are perfectly legal features, but if the model leans entirely on them, its "insight" is just restating scale, not finding SEO opportunity.

**The Test:** We must train the model *with* vs. *without* March volume metrics (`total_impressions` and `total_clicks`). We need to see how much the model's advantage shrinks when we remove these "close cousin" scale features, proving whether it is actually learning nuanced SEO patterns or just acting as a simple traffic forecaster.

In [27]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# 1. Create a restricted feature list (No Impressions, No Clicks)
restricted_features = ['log_engaged_sessions_march', 'ctr_march', 'average_position_march']

# 2. Re-slice our training and testing data using only the restricted features
X_train_restricted = X_train[restricted_features]
X_test_restricted = X_test[restricted_features]

# 3. Retrain the Linear Regression model blindfolded
sens_model = LinearRegression()
sens_model.fit(X_train_restricted, y_train)

# 4. Get predictions and reverse the log transform
sens_pred_log = sens_model.predict(X_test_restricted)
sens_pred_real = np.expm1(sens_pred_log)

# 5. Build our Results dataframe for the restricted model
sens_results_df = eligible_df.iloc[test_idx][['content_hash_id', 'missed_clicks_april', 'ctr_april', 'tier_avg_ctr_april']].copy()
sens_results_df['predicted_missed_clicks'] = sens_pred_real

# 6. Sort descending and grab Top 50
sens_ranked = sens_results_df.sort_values(by='predicted_missed_clicks', ascending=False)
sens_top_50 = sens_ranked.head(50).copy()

# 7. Evaluate the Business Metrics
sens_top_50['is_actual_opportunity'] = sens_top_50['ctr_april'] < (0.5 * sens_top_50['tier_avg_ctr_april'])
sens_precision = sens_top_50['is_actual_opportunity'].mean()
sens_total_captured = sens_top_50['missed_clicks_april'].sum()

print("--- SENSITIVITY CHECK: NO VOLUME METRICS ---")
print(f"Restricted Model Precision@50: {sens_precision * 100:.1f}%")
print(f"Restricted Model Captured:     {sens_total_captured:,.0f} clicks")

print("\n--- WHAT DID WE LEARN? ---")
print(f"Value lost by hiding traffic data: {(total_captured_clicks - sens_total_captured):,.0f} clicks")


--- SENSITIVITY CHECK: NO VOLUME METRICS ---
Restricted Model Precision@50: 76.0%
Restricted Model Captured:     179 clicks

--- WHAT DID WE LEARN? ---
Value lost by hiding traffic data: 1,096 clicks


### 4. Errors, Interpretation, and Final Verdict

**The Scoreboard (Test Set Evaluation)**
*   **Baseline (W04 Rule):** 94.0% Precision@50 | 2,131 Total Captured Clicks
*   **Linear Regression (LR):** 96.0% Precision@50 | 1,274 Total Captured Clicks
*   **Random Forest (RF):** 86.0% Precision@50 | 1,698 Total Captured Clicks

**1. The Coefficient Sign Issue (Multicollinearity)**
When reading the Linear Regression weights, `average_position_march` returned a negative coefficient (-0.0340). Standard SEO logic suggests a positive relationship (worse position = more missed opportunity). However, this is a classic symptom of multicollinearity. Because `ctr_march` and `log_impressions` are already in the model, they fully absorb the signal of position. The model learned that once you control for CTR and impressions, a deeper rank actually mathematically limits the ceiling for total missed clicks.

**2. Why Linear Regression Underperformed the Baseline**
The LR model achieved stellar Precision (96.0%) but failed on the metric that actually matters: Business Value (Total Captured Clicks). This is a direct consequence of the `log1p` transformation. While logging the target was mathematically necessary to stabilize the model and prevent a few giant "whale" pages from dominating the squared error loss, it caused the model to be less aggressive. By shrinking the massive gaps, the model learned to prioritize "safe" medium-sized opportunities rather than aggressively hunting the massive traffic whales that the Baseline naturally surfaces.

**3. The Random Forest Trade-off**
The Random Forest model closed a significant portion of the value gap (capturing 1,698 clicks), proving that a more flexible, non-linear model handles the log-compressed target much better than a single global linear formula. Interestingly, the RF captured more total clicks than the LR, but at the cost of lower Precision@50 (86.0% vs 96.0%). This highlights a very real precision vs. value trade-off: capturing the biggest whales often requires accepting a few more "noisy" or borderline recommendations in the Top 50.

**4. The Single-Split Limitation (Client Concentration)**
It is critical to acknowledge that 78% (39 out of 50) of the model's top predictions came from a single client. Because we used a single `GroupShuffleSplit`, this result is heavily skewed by the specific clients that happened to land in the test set. Therefore, these results are directional, not fully validated. Implementing a full `GroupKFold` cross-validation across multiple splits is the necessary and honest next step to prove generalizability.

**5. The Volume Sensitivity Check**
To test if the model was actually learning nuanced SEO patterns, we ran a sensitivity check by retraining the LR model blindfolded—removing `log_impressions_march` and `log_clicks_march`. The model's performance completely crashed, capturing only 179 clicks. This connects directly back to the multicollinearity finding in Point 1: our five features aren't five independent signals. Volume and CTR-related columns are highly entangled, and once you strip volume out, there's not much genuinely independent SEO signal left in what remains. The model's signal is almost entirely volume-driven; `ctr_march`, `log_engaged_sessions_march`, and `average_position_march` alone carry very little predictive power for which pages will have big April opportunities.

**Final Verdict**
In this experiment, the Machine Learning models did not beat the simple Baseline heuristic on the metric that ultimately matters: raw business value (2,131 captured clicks for Baseline vs 1,274 for LR and 1,698 for RF). A sensitivity check confirmed the model's signal is almost entirely volume-driven — removing impression/click features collapsed captured value by 86% (1,274 → 179 clicks for LR) — meaning the model has not found independent SEO insight beyond 'big pages stay big,' which is consistent with the baseline's simpler, volume-weighted approach already capturing most of the real signal. Because the problem is fundamentally algebraic (CTR Gap × Volume), the Baseline heuristic remains the recommended production system—it is more performant, perfectly interpretable, and requires zero ML infrastructure. However, the upward trend in value captured from Linear Regression to Random Forest is a real, promising signal, proving that more flexible models are worth investigating in future iterations.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.